In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import tqdm

import sys
sys.path.append("/home/max/Repos/Magnetic-Reconnection-Visualization/src/")

from mrvis.utils import print_time

CMAP = plt.get_cmap("cmr.lavender")


def get_ordered_list_of_times(timesteps):
    """Get ordered list of time steps from dings.h5

    Parameters
    ----------
    timesteps : List
        List of timesteps of type string

    Returns
    -------
    np.array
        Sorted list of timestep strings
    """
    times = np.zeros(len(timesteps), dtype="int")
    for i, step in enumerate(timesteps):
        if "Timestep" not in step:
            continue
        times[i] = int(step.split("Timestep_")[-1])
    return np.sort(times)


def get_data_from_name(f, field_name, timestep):
    # Get current vector
    if len(field_name) == 3:
        field_h5 = f[timestep][field_name[0]][field_name[1]]

        # Convert to numpy array and crop
        vector = np.array(
            [
                field_h5[field_name[-1] + "[0]"],
                field_h5[field_name[-1] + "[1]"],
                field_h5[field_name[-1] + "[2]"],
            ]
        )

        scalar = None

    if len(field_name) == 2:
        field_h5 = f[timestep][field_name[0]]

        # Convert to numpy array and crop
        vector = np.array(
            [
                field_h5[field_name[-1] + "[1]"],
                field_h5[field_name[-1] + "[2]"],
                field_h5[field_name[-1] + "[3]"],
            ]
        )
        
    return vector

In [ ]:
python configure.py --prob my_problem --coord cartesian --eos adiabatic --flux hlld --nghost 2 --mpi --precision double --cxxflags "-O3 -march native" # any extra compiler flags


In [ ]:
FIELD_LIST = [
    ["felder", "E", "E"],
    ["felder", "B", "B"],
    ["rho", "rhoL"],
    ["rho", "rhoM"],
]

filepath = ""
name = "solar-wind"
savepath = 
resolution = (256, 256, 256)
plot = False

f = h5py.File(filepath, "r")

timesteps = list(f.keys())

times = get_ordered_list_of_times(timesteps)

# grid = pv.ImageData(dimensions=dset.shape, origin=origin, spacing=spacing)

# Iterate over all time steps
for i in tqdm.tqdm(range(1, len(times))):
    timestep = f"Timestep_{times[i]}"
    
    # Iterate over all fields
    field_numpy = []
    for field_name in FIELD_LIST:
        try:
            vector = get_data_from_name(f, field_name, timestep)
            
            # Save as vti
            field_numpy.append((vector, "vectors-" + field_name[-1], scalar, "scalars-" + field_name[-1]))

            # Plot slice in the middle of current vector
            if plot:
                field_slice = vector[:, :, :, vector.shape[-1] // 2]
                plt.imshow(np.linalg.norm(field_slice, axis=0), cmap=CMAP)
                plt.show()

        except Exception as ex:
            print(f"Step {i} with key {times[i]} and vector {field_name[-1]} failed:", ex)

        # Save the current time step as vti with pyvista
        grid = pv.ImageData(dimensions=resolution, origin=(0, 0, 0), spacing=(1, 1, 1))

        # Add all fields to the grid
        for idx in range(len(field_numpy)):
            vector, fname, scalar, sname = field_numpy[idx]
            vecs = vector.transpose(1, 2, 3, 0)
            vecs_flat = np.array([vecs[:, :, :, i].flatten(order="F") for i in range(3)]).T
            grid.point_data.set_vectors(vecs_flat, fname)
            if scalar is not None:
                grid.point_data.set_scalars(scalar.flatten(order="F"), sname)

        # Write the VTK file
        date = print_time()
        grid.save(savepath + f"{date}-{name}.{str(i).zfill(4)}.vti")
